In [1]:
import os
from dotenv import load_dotenv
import pandas as pd
import sys
sys.path.insert(0, "/Users/jihaein/Desktop/fds_main/investment-evidence-system")

In [2]:
from backtest.engine import load_daily_price
from backtest.montecarlo import run_montecarlo
import os; from dotenv import load_dotenv; load_dotenv()

base = os.getenv("BACKTEST_DATA_DIR")
grid = load_daily_price()
print("grid:", grid.shape, " base set:", bool(base))


grid: (1249, 120)  base set: True


In [4]:
from backtest.montecarlo import _verdict, DEFAULT_BASELINE_DATE
from models.seoseohi.pipeline import load_ksic_from_base
from models.seoseohi.fscore import list_snapshot_dates

ksic = load_ksic_from_base(base)
dates = [d for d in list_snapshot_dates(base) if d > DEFAULT_BASELINE_DATE]
for s in grid.columns:
    for d in dates:
        _verdict(base, ksic, s, DEFAULT_BASELINE_DATE, d)
        
print("cache warmed:", len(__import__('backtest.montecarlo', fromlist=['_VERDICT_CACHE'])._VERDICT_CACHE))

cache warmed: 2888


## 핵심 — 네 전략 비교 (2년 · 4년, 전체 지표)
무손절 / 단순손절 / 서서히(개수) / 서서히(합산 4점).
지표: 누적수익률 · 수익률_중앙값 · MDD · Sortino · Sharpe · 평균_현금대체 · 평균_매도수 · 적중률 · 승률.


In [ ]:
# 네 전략 비교 (서서히 개수 vs 서서히 합산 4점) — PM 2026-11-13 규칙 반영
# with_sum=True 면 '서서히_합산'(세 점수 합계>=4 매도) 전략이 추가된다.
for yrs in [2, 4]:
    detail, summary = run_montecarlo(grid, base, years=yrs, seed=0, with_sum=True)
    print(f"=== {yrs}년 (seed=0, 바스켓 10개) ===")
    print(summary.round(3).to_string())
    print()

## 아웃오브샘플 검증 — 엣지가 일반화되는가
규칙은 120종목·2016~2020 전체로 맞춰졌으므로, 아래는 **엄밀한 train/test가 아니라 '쏠림/안정성' 검사**다.
엣지가 두 종목 그룹·두 기간 모두에서 유지되면 견고하고, 한쪽에만 몰리면 과적합 신호다.
(진짜 out-of-sample은 데이터팀의 새 종목/기간이 필요하다.)


In [ ]:
# 아웃오브샘플 1: 종목 반분 — 엣지가 두 그룹 모두에서 유지되나?
import numpy as np
codes = sorted(grid.columns)
perm = np.random.default_rng(42).permutation(codes)
halves = {"A조 (60종목)": sorted(perm[:60]), "B조 (60종목)": sorted(perm[60:])}
for name, half in halves.items():
    _, s = run_montecarlo(grid[half], base, years=4, seed=0, n_pick=20, n_runs=10, with_sum=True)
    print(f"===== {name} · 4년 =====")
    print(s.round(3).to_string())
    print()

In [ ]:
# 아웃오브샘플 2: 다른 기간 — 2018년 진입(후반부)에서도 엣지가 유지되나?
for label, entry, baseline in [("2016 진입 (원래)",  "2016-01-04", "2016-04-01"),
                               ("2018 진입 (후반부)", "2018-01-02", "2018-04-01")]:
    _, s = run_montecarlo(grid, base, years=2, seed=0,
                          entry_date=entry, baseline_date=baseline, with_sum=True)
    print(f"===== {label} · 2년 =====")
    print(s.round(3).to_string())
    print()


## 매도 품질 — 종목별 · 점수별, 그리고 정직한 집계
아래 표는 120종목 전체 매도와 D/F/O 점수. 그 다음 셀은 078130 같은 **희석 이상치**가
평균을 왜곡하므로 원시 · winsorize · 중앙값을 함께 보여준다.


In [ ]:
# 120종목 전체 서서히 매도 표 (개수 방식 / 합산 4점 비교, 세 모델 점수 포함) — PM 2026-11-13
import os, pandas as pd
from backtest.montecarlo import (seoseohi_schedule, _scores,
                                 DEFAULT_BASELINE_DATE, DEFAULT_ENTRY_DATE, DEFAULT_TRIGGER)
from models.seoseohi.pipeline import integrate, integrate_sum, load_ksic_from_base
from models.seoseohi.fscore import list_snapshot_dates
from backtest.engine import CASH_DAILY_RATE

YEARS = 4                      # None 이면 2020-12 까지
MODES = ["count", "sum"]
GOOD, BAD = "✅ 좋은매도", "❌ 성급한매도"

ksic = load_ksic_from_base(base)
entry = pd.Timestamp(DEFAULT_ENTRY_DATE)
end_date = None if YEARS is None else entry + pd.DateOffset(years=YEARS)

nm = pd.read_csv(os.path.join(base, "12_기업개황_KSIC.csv"),
                 header=0, skiprows=[1], dtype=str, encoding="utf-8-sig")
nm["stock_code"] = nm["stock_code"].str.zfill(6)
name_map = dict(zip(nm["stock_code"], nm["corp_name"]))

all_codes = list(grid.columns)
eval_dates = [d for d in list_snapshot_dates(base)
              if pd.Timestamp(d) > pd.Timestamp(DEFAULT_BASELINE_DATE)
              and (end_date is None or pd.Timestamp(d) <= pd.Timestamp(end_date))]

def _fmt(x):
    return "-" if pd.isna(x) else str(int(x))

def r_between(stock, d0, d1):
    seg = grid.loc[pd.Timestamp(d0):pd.Timestamp(d1), stock].dropna().iloc[1:]
    return (1 + seg / 100).prod() - 1 if len(seg) else float("nan")

for mode in MODES:
    integ = integrate if mode == "count" else integrate_sum
    sched = seoseohi_schedule(base, ksic, all_codes, grid, DEFAULT_BASELINE_DATE,
                              DEFAULT_TRIGGER, end_date, price_gate=False, mode=mode)
    rows = []
    for s, sell_date in sched.items():
        sc = None
        for d in eval_dates:                       # 매도를 일으킨 기준일의 점수 찾기
            cand = _scores(base, ksic, s, DEFAULT_BASELINE_DATE, d)
            if integ(cand) in DEFAULT_TRIGGER:
                sc = cand
                break
        if sc is None:
            continue
        fut = grid.loc[sell_date:, s]
        if end_date is not None:
            fut = fut.loc[:pd.Timestamp(end_date)]
        fut = fut.dropna().iloc[1:]
        if len(fut) == 0:
            continue
        R_hold = (1 + fut / 100).prod() - 1
        R_cash = (1 + CASH_DAILY_RATE) ** len(fut) - 1
        d_, f_, o_ = sc.get("dupont"), sc.get("fscore"), sc.get("ohlson")
        tot = sum(int(v) for v in (d_, f_, o_) if not pd.isna(v))
        sig = "+".join(f"{m}{_fmt(v)}" for m, v in (("D", d_), ("F", f_), ("O", o_))
                       if not pd.isna(v) and int(v) > 0)
        rows.append({"매도일": pd.Timestamp(sell_date).date(), "stock": s,
                     "종목명": name_map.get(s, ""), "매수~매도": r_between(s, entry, sell_date),
                     "판정": GOOD if (R_cash - R_hold) > 0 else BAD, "매도후": R_hold,
                     "D": _fmt(d_), "F": _fmt(f_), "O": _fmt(o_), "합계": tot, "신호": sig})
    if not rows:
        print(f"[{mode}] 매도 없음"); continue
    tbl = pd.DataFrame(rows).sort_values("매도일")
    g, b = (tbl["판정"] == GOOD).sum(), (tbl["판정"] == BAD).sum()
    print(f"\n===== {mode} 방식 : 매도 {len(tbl)}건, 좋은 {g} / 성급 {b}, 적중률 {g / max(g + b, 1):.1%} =====")
    show = tbl.copy()
    show["매수~매도"] = show["매수~매도"].map(lambda x: f"{x:+.1%}")
    show["매도후"] = show["매도후"].map(lambda x: f"{x:+.1%}")
    cols = ["매도일", "stock", "종목명", "매수~매도", "판정", "매도후", "D", "F", "O"]
    cols += (["합계"] if mode == "sum" else []) + ["신호"]
    print(show[cols].to_string(index=False))
    print(f"\n[{mode}] 신호(점수) 조합별 성적:")
    agg = tbl.groupby("신호").agg(건수=("stock", "size"),
                                적중률=("판정", lambda x: (x == GOOD).mean()),
                                매도후중앙=("매도후", "median"))
    agg["적중률"] = agg["적중률"].map(lambda x: f"{x:.0%}")
    agg["매도후중앙"] = agg["매도후중앙"].map(lambda x: f"{x:+.1%}")
    print(agg.sort_values("건수", ascending=False).to_string())


In [ ]:
# 매도 품질 집계: 원시 vs winsorize(이상치 중화) vs 중앙값 — 정직한 보고
import pandas as pd
from backtest.montecarlo import sell_study

ss = sell_study(grid, base, ksic, years=4, seed=0, mode="sum")   # 합산 4점 기준
u = ss.drop_duplicates("stock")                                  # 고유 매도 종목
rh, B = u["R_hold"], u["B"]
B_wins = B + (rh - rh.clip(upper=1.0))                           # R_hold 을 +100%로 캡했을 때의 B

print(f"고유 매도 {len(u)}건  |  적중률 {u['good'].mean():.0%}")
print(f"평균B   원시 {B.mean():+.3f}   winsorize {B_wins.mean():+.3f}   |   중앙B {B.median():+.3f}")
print(f"매도후 수익률   평균 {rh.mean():+.1%}   중앙 {rh.median():+.1%}")
print("\n※ 평균은 078130 같은 희석 이상치 한 건에 휘둘린다. 중앙값·winsorize가 대표적인 매도의 실제 가치.")


In [6]:
# ── F-Score 1점 드롭 실험 (파일 영구변경 없이, 이전 패치도 자동 복구) ──
import backtest.montecarlo as mc
import pandas as pd
from models.seoseohi.pipeline import integrate_sum as _orig_sum   # 진짜 원본
mc.integrate_sum = _orig_sum          # 이전 실험이 남긴 패치가 있으면 여기서 복구

def _sum_nof1(scores):
    s = dict(scores)
    fs = s.get("fscore")
    if pd.notna(fs) and int(fs) == 1:     # F-Score 1점만 합계에서 제외 (NA/None 은 그대로)
        s["fscore"] = 0
    return _orig_sum(s)

def _show(tag):
    print(f"[{tag}] 서서히_합산")
    for label, e, b in [("2016 진입·2년", "2016-01-04", "2016-04-01"),
                        ("2018 진입·2년", "2018-01-02", "2018-04-01")]:
        _, s = mc.run_montecarlo(grid, base, years=2, seed=0,
                                 entry_date=e, baseline_date=b, with_sum=True)
        r = s.loc["서서히_합산"]
        print(f"  {label}: 수익 {r['누적수익률']:+.3f}  MDD {r['MDD']:.3f}  "
              f"Sharpe {r['Sharpe']:.3f}  승률 {r['win_vs_무손절']:.2f}")
    ss = mc.sell_study(grid, base, ksic, years=4, seed=0, mode="sum").drop_duplicates("stock")
    print(f"  4년 고유매도 {len(ss)}건  적중률 {ss['good'].mean():.0%}  "
          f"중앙B {ss['B'].median():+.3f}  최악_매도후 {ss['R_hold'].max():+.1%}")

_show("기존 (F1 포함)")
mc.integrate_sum = _sum_nof1
print()
_show("F1 드롭")
mc.integrate_sum = _orig_sum
print("\n원복 완료")

[기존 (F1 포함)] 서서히_합산
  2016 진입·2년: 수익 +0.176  MDD 0.181  Sharpe 0.540  승률 0.80
  2018 진입·2년: 수익 +0.131  MDD 0.327  Sharpe 0.238  승률 0.70
  4년 고유매도 60건  적중률 67%  중앙B +0.108  최악_매도후 +154.6%

[F1 드롭] 서서히_합산
  2016 진입·2년: 수익 +0.177  MDD 0.181  Sharpe 0.540  승률 0.70
  2018 진입·2년: 수익 +0.114  MDD 0.334  Sharpe 0.217  승률 0.60
  4년 고유매도 55건  적중률 71%  중앙B +0.132  최악_매도후 +132.6%

원복 완료
